# 슈카월드 컨텐츠를 기반으로 채팅하기







## Semantic Searching
  - 데이터를 로드합니다.
  - 질문의 Embedding 을 계산합니다.
  - 계산된 Embedding 과 유사한 결과들을 탐색합니다.
  - (참조) https://platform.openai.com/docs/guides/embeddings/what-are-embeddings

In [1]:
!pip install tiktoken
!pip install openai

   ---------------------------------------- 0.0/883.8 kB ? eta -:--:--
   --------------------------------------- 883.8/883.8 kB 19.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 36.7 MB/s eta 0:00:00


## 준비!

라이브러리 들을 읽고, API KEY 를 설정합니다.

In [ ]:
import numpy as np
import pandas as pd
import openai
import tiktoken
import os

In [ ]:
os.environ['OPENAI_API_KEY'] = "" # OpenAI api key를 입력하세요(문자열)

client = openai.OpenAI()

### Preview

OpenAI의 API를 사용하여 GPT 를 사용합니다.

In [ ]:
prompt = "우크라이나와 러시아의 전쟁은 어떻게 되어가고 있나요?"

# GPT-4로 답변 생성
completion = client.chat.completions.create(
  model="gpt-4",
  messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
  ]
)

# API 응답에서 답변 추출
print(completion.choices[0].message.content)

## 슈카월드 데이터 읽기

준비된 데이터를 읽습니다.
영상에 대한 내용 (글) 과 이를 임베딩하여 벡터로 치환한 값들이 저장되어 있습니다.

In [ ]:
datafile_path = "./data/shuka_embeddings.csv"

df = pd.read_csv(datafile_path)
df["embedding"] = df.embedding.apply(eval).apply(np.array)
df

## 텍스트 임베딩

OPENAI 의 api 를 이용하여 프롬프트를 임베딩 해봅시다.

In [ ]:
emb = client.embeddings.create(
    input=prompt,
    model='text-embedding-ada-002')
emb

## 임베딩으로 이용한 검색

임베딩을 만들어서 준비된 데이터와 임베딩을 비교합니다.  
cosine 을 계산하여 벡터간 거리를 측정합니다.  
가장 가까운 순으로 정렬하여 검색된 데이터를 관찰해봅니다.  

OpenAI 에서는 이를 Sementic Search 라고 부릅니다.

나만의 검색엔진, 추천 시스템이 구축되었습니다.

In [ ]:
from scipy.spatial.distance import cosine

# 연관된 영상을 검색합니다.
def search_video(df, question, n=3, pprint=True):
    # 질문에 대한 임베딩 생성
    q_embeddings = client.embeddings.create(input=question, model='text-embedding-ada-002').data[0].embedding
    
    df["distance"] = df["embedding"].apply(lambda x: cosine(q_embeddings, x))
    
    results = (
        df.sort_values("distance", ascending=True)
        .head(n)
    )

    return results

In [ ]:
# 하고 싶은 말을 입력하고, 비슷한 결과들을 열람합니다.
results = search_video(df, prompt, n=5)
results

## 검색 결과와 함께 질문하기!
  - 질문과 유사한 컨텐츠를 검색합니다.
  - GPT 의 프롬프트에 유사한 결과를 함께 넣어 줍니다.
  - GPT 가 학습할 때 없었던 정보이지만, 이렇게 같이 넣어주면 GPT 가 누락되지 않은 정보를 이용하여 대답을 할 수 있습니다.

In [ ]:
GPT_MODEL="gpt-4"
def num_tokens(text: str, model: str = GPT_MODEL) -> int:
    """Return the number of tokens in a string."""
    encoding = tiktoken.encoding_for_model(model)
    return len(encoding.encode(text))


def query_message(
    query: str,
    df: pd.DataFrame,
    model: str,
    token_budget: int
) -> str:
    """Return a message for GPT, with relevant source texts pulled from a dataframe."""
    
    results = search_video(df, query, n=3)
    strings = results['combined']
    
    introduction = 'Use the below articles on the 슈카월드 to answer the subsequent question. If the answer cannot be found in the articles, write "I could not find an answer."'
    question = f"\n\nQuestion: {query}"
    message = introduction
    for string in strings:
        next_article = f'\n\n:\n"""\n{string}\n"""'
        print("Tokens in Use : ", num_tokens(message + next_article + question, model=model))
        if (
            num_tokens(message + next_article + question, model=model)
            > token_budget
        ):
            
            break
        else:
            message += next_article
    return message + question

def ask(
    query: str,
    df: pd.DataFrame = df,
    model: str = GPT_MODEL,
    token_budget: int = 8192 - 500,
    print_message: bool = False,
) -> str:
    """Answers a query using GPT and a dataframe of relevant texts and embeddings."""
    message = query_message(query, df, model=model, token_budget=token_budget)
    if print_message:
        print(message)
        
    completion = client.chat.completions.create(
      model=model,
      messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": message}
      ]
    )

    # API 응답에서 답변 추출
    return completion.choices[0].message.content

In [ ]:
# set print_message=True to see the source text GPT was working off of
answer = ask('우크라이나와 러시아의 전쟁은 어떻게 되어가고 있나요?', model="gpt-4", print_message=True)

In [ ]:
answer